# Tarea 3

**Autor:** Alejandro Zarate Macias  

**Curso:** Reconocimiento de Patrones (ML)

**Fecha:** 16 de Febrero 2026

---

## Introducción

Esta tarea avanza el estudio del aprendizaje supervisado explorando modelos más complejos como Redes Neuronales y Máquinas de Vectores de Soporte (SVM). Se implementan ambos modelos en problemas de regresión y clasificación, tanto en conjuntos de datos reales como en funciones analíticas. Se analiza el proceso de backpropagation, se comparan con métodos de aproximación de gradientes, y se evalúa el desempeño de ambos modelos bajo diferentes hiperparámetros y configuraciones. Finalmente, se examina la interpretabilidad de los modelos y la importancia de la regularización en la generalización.

---

## Pre-requisitos

- Python 3.9 o superior
- Librerías:
    - numpy
    - matplotlib
    - notebook
    - pandas
    - scikit-learn
    - kagglehub[pandas-datasets]
    - ipywidgets

---

# Problema 2

Considere el conjunto de datos de `2022 Fuel Consumption Ratings`. Cree un script de Python para resolver el problema de regresión de pronóstico del consumo de combustible en ciudad y carretera con `sklearn`. Utilice un solo modelo `NN` para las dos categorías de consumo de combustible. Anote todos los supuestos, hiperparámetros y operaciones de preprocesamiento de datos que realice.

### Librerías

In [ ]:
import pandas as pd
import kagglehub
import matplotlib.pyplot as plt
from kagglehub import KaggleDatasetAdapter
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score

### Carga de datos

In [ ]:
file_path = "MY2022 Fuel Consumption Ratings.csv"

df: pd.DataFrame = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "rinichristy/2022-fuel-consumption-ratings",
    file_path
)

In [ ]:
df.head()

### Preprocesamiento de Datos

In [ ]:
# Seleccion de features y variables objetivo

# X:
# Variables numericas: 'Engine Size(L)', 'Cylinders'
# Variables categoricas: 'Vehicle Class', 'Transmission', 'Fuel Type'
X = df[['Engine Size(L)', 'Cylinders', 
        'Vehicle Class', 'Transmission', 'Fuel Type']]

# Manejo de variables categoricas con get_dummies
X = pd.get_dummies(X, columns=['Vehicle Class', 'Transmission', 'Fuel Type'], drop_first=True)


# y:
# Para el modelo de consumo en ciudad
y_city = df['Fuel Consumption (City (L/100 km)']

# Para el modelo de consumo en carretera
y_hwy = df['Fuel Consumption(Hwy (L/100 km))']

In [ ]:
# Split de datos en entrenamiento y prueba
X_train_city, X_test_city, y_train_city, y_test_city = train_test_split(X, y_city, test_size=0.2, random_state=14)
X_train_hwy, X_test_hwy, y_train_hwy, y_test_hwy = train_test_split(X, y_hwy, test_size=0.2, random_state=14)

# Tamaño de los conjuntos de entrenamiento y prueba
print(f"Conjunto de entrenamiento ciudad: {X_train_city.shape} muestras")
print(f"Conjunto de prueba ciudad: {X_test_city.shape} muestras")
print(f"Conjunto de entrenamiento carretera: {X_train_hwy.shape} muestras")
print(f"Conjunto de prueba carretera: {X_test_hwy.shape} muestras")

### Entrenamiento

In [ ]:
# Modelo de ciudad
model_city = MLPRegressor(
    loss='squared_error',
    hidden_layer_sizes=(200,), 
    activation='relu', 
    solver='adam', 
    max_iter=1500, 
    tol=1e-4,
    random_state=14
)
model_city.fit(X_train_city, y_train_city)
print("Modelo de Ciudad entrenado.")

In [ ]:
# Modelo de carretera
model_hwy = MLPRegressor(
    loss='squared_error',
    hidden_layer_sizes=(200,), 
    activation='relu', 
    solver='adam', 
    max_iter=1500,
    tol=1e-4,
    random_state=14
)
model_hwy.fit(X_train_hwy, y_train_hwy)
print("Modelo de Carretera entrenado.")

### Evaluación

In [ ]:
# Metricas modelo de ciudad

y_pred_city = model_city.predict(X_test_city)
mse_city = mean_squared_error(y_test_city, y_pred_city)
rmse_city = mse_city ** 0.5
r2_city = r2_score(y_test_city, y_pred_city)

# Metricas modelo carretera

y_pred_hwy = model_hwy.predict(X_test_hwy)
mse_hwy = mean_squared_error(y_test_hwy, y_pred_hwy)
rmse_hwy = mse_hwy ** 0.5
r2_hwy = r2_score(y_test_hwy, y_pred_hwy)

# Tabla de resultados
print('-' * 60)
print(f"{'Modelo':<15} {'MSE':<15} {'RMSE':<15} {'R²':<15}")
print('-' * 60)
print(f"{'Ciudad':<15} {mse_city:<15.4f} {rmse_city:<15.4f} {r2_city:<15.4f}")
print(f"{'Carretera':<15} {mse_hwy:<15.4f} {rmse_hwy:<15.4f} {r2_hwy:<15.4f}")
print('-' * 60)

In [ ]:
# Gráficas de predicciones vs valores reales
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfica para Ciudad
axes[0].scatter(y_test_city, y_pred_city, alpha=0.7)
axes[0].plot([y_test_city.min(), y_test_city.max()], [y_test_city.min(), y_test_city.max()], 'r--')
axes[0].set_xlabel('Valores Reales (Ciudad)')
axes[0].set_ylabel('Predicciones (Ciudad)')
axes[0].set_title('Predicciones vs Valores Reales - Ciudad')
axes[0].grid()

# Gráfica para Carretera
axes[1].scatter(y_test_hwy, y_pred_hwy, alpha=0.7)
axes[1].plot([y_test_hwy.min(), y_test_hwy.max()], [y_test_hwy.min(), y_test_hwy.max()], 'r--')
axes[1].set_xlabel('Valores Reales (Carretera)')
axes[1].set_ylabel('Predicciones (Carretera)')
axes[1].set_title('Predicciones vs Valores Reales - Carretera')
axes[1].grid()

plt.tight_layout()
plt.show()

---

# Problema 3

Considere la siguiente función:

$f(x) = 1-2^{\sin(-x^2)}, \quad x \in \mathcal{I}[-\pi, \pi]$

Crea un script en Python para resolver el problema de regresión asociado a una red neuronal con sklearn. Anota todas las suposiciones que hagas. También anota los hiperparámetros que elijas y explica por qué los elegiste así.

### Librerías

### Carga de datos

### Preprocesamiento de Datos

### Entrenamiento

### Evaluación